# 점자 Transformer 코드 읽기와 실행

출력은 비워져 있다. 말뭉치와 checkpoint는 포함하지 않는다. 이 노트북은 최종 16 epoch 기준 실험의 소스 흐름을 연결한다. 합성 테스트 문자열은 실제 점역 예제가 아니다.

In [ ]:
from pathlib import Path
import json, sys, subprocess

# 프로젝트 루트 또는 notebooks/ 폴더에서 실행한다.
ROOT = Path.cwd().resolve()
if not (ROOT / "README.md").is_file():
    ROOT = ROOT.parent
assert (ROOT / "configs").is_dir(), "프로젝트 폴더에서 JupyterLab을 실행해야 합니다."

# 실행 플래그가 False이면 자료를 읽거나 모델을 로딩하지 않는다.
RUN_TRAINING = False
RUN_EVALUATION = False
RUN_APP = False


In [ ]:
print(json.dumps(json.loads((ROOT / "configs/reference_run.json").read_text(encoding="utf-8")), ensure_ascii=False, indent=2))

## 모델 구조

`model.py`의 Vocabulary는 train 문자만 사전에 등록한다. NFD는 한글을 자모로 분해해 작은 출력 어휘를 만든다. Embedding은 문자 ID를 학습 벡터로 바꾼다. Encoder self-attention은 점자 문맥을 읽고, decoder causal self-attention은 이전 출력만 참조한다. Cross-attention은 encoder의 점자 표현을 참조한다. FFN은 각 위치의 특징을 비선형 변환하고, LayerNorm과 잔차 연결은 층별 계산을 안정화한다.

256차원을 8개 head로 나누면 head당 32차원이다. Encoder 3층과 decoder 3층은 서로 다른 파라미터를 갖는다. 위치 신호는 고정 sin/cos 공식이다.

In [ ]:
print((ROOT / "model.py").read_text(encoding="utf-8"))

## 학습 루프

`train.py`는 정답 앞부분으로 다음 정답 토큰을 예측하는 teacher forcing을 사용한다. Cross entropy는 정답의 음의 log 확률이다. backward는 연쇄법칙으로 gradient를 구하고 AdamW는 가중치를 갱신한다. PAD는 loss에서 제외한다. 여러 batch의 gradient를 모을 때 유효 토큰 수로 가중한다. 학습 길이 초과는 기본 오류이며 자동 잘림이 아니다.

자료 준비 명령은 README에 있다. 다음 셀은 실행 플래그를 True로 바꾸면 전체 학습을 시작한다.

In [ ]:
if RUN_TRAINING:
    subprocess.run([sys.executable, "train.py", "--run-dir", "checkpoints/reference", "--device", "cuda", "--epochs", "16", "--batch-size", "32", "--grad-accum", "1"], cwd=ROOT, check=True)
else:
    print("학습 실행 안 함")

## 평가와 추론

Teacher forcing loss와 실제 생성 성능은 다르다. 평가는 정답 접두사 없이 BOS부터 한 글자씩 생성한다. 잘못 출력한 토큰도 다음 생성의 조건이 된다. 저장된 best.pt, config.json, vocab.json은 같은 실험 묶음이어야 한다. CPU 앱은 gradient 계산을 끄고 저장 가중치를 읽는다. GPU는 CPU 추론의 필수 조건이 아니다.

In [ ]:
if RUN_EVALUATION:
    subprocess.run([sys.executable, "evaluate.py", "--checkpoint", "checkpoints/reference", "--device", "cuda"], cwd=ROOT, check=True)
if RUN_APP:
    subprocess.Popen([sys.executable, "app.py", "--checkpoint", "checkpoints/reference"], cwd=ROOT)
print(json.dumps(json.loads((ROOT / "results/reference_metrics.json").read_text(encoding="utf-8")), ensure_ascii=False, indent=2))